# Restricted-input method reference
Source: analysis/intra-park/proximity_calculation/water_proximity.ipynb
Requires independently obtained inputs and source directory layout. Not an end-to-end pipeline. Outputs can contain sensitive data; keep the working directory private. External absolute paths have been replaced by external_inputs/<basename>. See docs/REPRODUCIBILITY.md.


In [ ]:
import os
from pathlib import Path
private_root = Path(os.environ["UFUG_PRIVATE_ROOT"]).resolve()
os.chdir(private_root)


In [ ]:
import geopandas as gpd
import numpy as np
# --- Setup: run this cell ---
import pandas as pd
from shapely.ops import unary_union, nearest_points
from shapely.geometry import LineString
from shapely.ops import unary_union
from sklearn.cluster import DBSCAN
from dataclasses import dataclass
from typing import Dict, Any, Literal, Optional

import os
# Working directory is explicitly configured in the first cell.


In [ ]:
EPSG_SG = 3414  # SVY21 / Singapore TM (metres)

def load_points(points_path: str) -> gpd.GeoDataFrame:
    """
    Reads points from CSV (lon,lat in WGS84) or a Point GeoJSON.
    Returns a GeoDataFrame in EPSG:3414.
    """
    p = str(points_path).lower()
    if p.endswith((".geojson", ".json", ".gpkg", ".shp")):
        pts = gpd.read_file(points_path)
        if pts.crs is None:
            # Assume WGS84 if missing
            pts = pts.set_crs(4326)
    else:
        # CSV expected with columns: lon, lat (WGS84)
        df = pd.read_csv(points_path)
        if not {"lon", "lat"}.issubset(df.columns):
            raise ValueError("CSV must contain 'lon' and 'lat' columns.")
        pts = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df["lon"], df["lat"]),
            crs=4326
        )
    # Reproject to EPSG:3414 for metric distances
    return pts.to_crs(EPSG_SG)

def load_water(water_path: str) -> gpd.GeoSeries:
    """
    Reads polygon/multipolygon waterbody and returns a single dissolved geometry in EPSG:3414.
    """
    water = gpd.read_file(water_path)
    if water.empty:
        raise ValueError("Waterbody file has no features.")
    if water.crs is None:
        water = water.set_crs(4326)
    water_3414 = water.to_crs(EPSG_SG)
    # Dissolve to one geometry for distance checks
    return gpd.GeoSeries([unary_union(water_3414.geometry.values)], crs=EPSG_SG)

def distances_to_water(water_series_3414: gpd.GeoSeries, pts_3414: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Computes Euclidean distance (m) from each point to the waterbody.
    Also returns the nearest point on the water edge (both EPSG:3414 and WGS84).
    """
    water_geom = water_series_3414.iloc[0]

    # Vectorised distances in metres
    dists_m = pts_3414.geometry.distance(water_geom)

    # Nearest point on water geometry (loop is fine for typical sizes)
    near_x, near_y = [], []
    for pt in pts_3414.geometry:
        _, q = nearest_points(pt, water_geom)   # q is on water
        near_x.append(q.x)
        near_y.append(q.y)

    out = pts_3414.copy()
    out["distance_m"] = dists_m.values
    out["nearest_x_m"] = near_x
    out["nearest_y_m"] = near_y

    # Also give nearest point in lon/lat for reference
    near_gdf = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(near_x, near_y),
        crs=EPSG_SG
    ).to_crs(4326)
    out["nearest_lon"] = near_gdf.geometry.x.values
    out["nearest_lat"] = near_gdf.geometry.y.values

    return out


In [ ]:
for park in ["jurong", "amk", "punggol"]:
    water_path = "analysis/intra-park/proximity_calculation/waterbody_sg.geojson"
    points_path = f"analysis/path_calculation/origins_{park}_coords.csv"    # CSV with lon,lat OR a Point GeoJSON
    water_3414 = load_water(water_path)
    pts_3414 = load_points(points_path)
    result_gdf = distances_to_water(water_3414, pts_3414)
    result_gdf.drop(columns="geometry").to_csv(f"analysis/intra-park/proximity_calculation/points_with_distance_{park}.csv", index=False)
    result_gdf.head()

In [ ]:
for park in ["jurong", "amk", "punggol"]:
    gdf_base = gpd.read_file(f'gis_files/hexagons/nearest_mrt/parks_nearest_mrt_{park}.geojson')
    result_gdf = pd.read_csv(f"analysis/intra-park/proximity_calculation/points_with_distance_{park}.csv")
    result_gdf = result_gdf[["grid_id", "distance_m"]]
    gdf_new = gdf_base.merge(result_gdf, on='grid_id', how='left')
    # inverted distance for easier interpretation (higher = closer) and for distance = 0, then set to 1
    gdf_new['inv_distance_m'] = np.where(gdf_new['distance_m'] > 0, 1 / gdf_new['distance_m'], 1)
    # summary stats

    mean_dist = gdf_new['inv_distance_m'].mean()
    median_dist = gdf_new['inv_distance_m'].median()
    std_dist = gdf_new['inv_distance_m'].std()
    print(f"{park} - Mean distance to water: {mean_dist:.3f} m, Median: {median_dist:.3f} m, Std Dev: {std_dist:.3f} m")
    gdf_new.to_file(f'analysis/intra-park/proximity_calculation/parks_nearest_water_{park}.geojson', driver='GeoJSON')